# CNN 卷积神经网络（上）

相比于上一节做的改变只有把线性神经网络改成卷积神经网络，把模型迁移到GPU，把输入和标签张量也迁移到GPU

In [1]:
import torch
from torchvision import transforms
from torchvision import datasets
from torch.utils.data import DataLoader
import torch.nn.functional as F
import torch.optim as optim

In [2]:
batch_size  = 64
transform = transforms.Compose([ 
    transforms.ToTensor(), 
    transforms.Normalize((0.1307,), (0.3081,)),
])

train_dataset = datasets.MNIST(
    '../mnist', 
    train=True, 
    download=True, 
    transform=transform) 
train_loader = DataLoader(
    train_dataset, 
    batch_size=batch_size, 
    shuffle=True)

test_dataset = datasets.MNIST(
    '../mnist', 
    train=False, 
    download=True, 
    transform=transform)
test_loader = DataLoader(
    test_dataset, 
    batch_size=batch_size, 
    shuffle=True)

In [3]:
class Net(torch.nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.conv1 = torch.nn.Conv2d(1,10,kernel_size=5) # 输入1个通道输出10个通道，卷积核5*5
        self.conv2 = torch.nn.Conv2d(10,20,kernel_size=5)
        self.pooling = torch.nn.MaxPool2d(2) # 池化核2*2
        self.fc = torch.nn.Linear(320,10) # 线性层，输出10类
        # 320是最终算出的张量为20通道4*4大小的元素个数

    def forward(self, x):
        batch_size = x.size(0) # 输入x的第一个维度是batchsize即输入个数
        x = self.pooling(F.relu(self.conv1(x)))
        x = self.pooling(F.relu(self.conv2(x)))
        x = x.view(batch_size, -1) # -1表示自动适配=元素总数/batch_size
        # 等价于x = x.flatten(start_dim=1)
        x = self.fc(x)
        return x

model = Net()
# 准备模型迁移到显卡
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model.to(device)

Net(
  (conv1): Conv2d(1, 10, kernel_size=(5, 5), stride=(1, 1))
  (conv2): Conv2d(10, 20, kernel_size=(5, 5), stride=(1, 1))
  (pooling): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc): Linear(in_features=320, out_features=10, bias=True)
)

In [4]:
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.5)

In [5]:
def train(epoch):
    running_loss = 0.0
    for batch_idx, data in enumerate(train_loader, 0):
        inputs, labels = data
        # 输入和label也迁移到GPU
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        if batch_idx % 300 == 299:
            print('[%d, %5d] loss: %.3f' % (epoch + 1, batch_idx + 1, running_loss / 300))
            running_loss = 0.0

In [6]:
def test():
    correct = 0
    total = 0
    with torch.no_grad():
        for data in test_loader:
            images, labels = data
            # 测试集的输入和label也迁移到GPU
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, dim=1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
        print("测试集的准确率：{}".format(correct / total))

In [7]:
%%time
if __name__=="__main__":
    for epoch in range(10):
        train(epoch)
        test()

[1,   300] loss: 0.651
[1,   600] loss: 0.196
[1,   900] loss: 0.141
测试集的准确率：0.9701
[2,   300] loss: 0.109
[2,   600] loss: 0.099
[2,   900] loss: 0.087
测试集的准确率：0.9764
[3,   300] loss: 0.079
[3,   600] loss: 0.073
[3,   900] loss: 0.071
测试集的准确率：0.9827
[4,   300] loss: 0.060
[4,   600] loss: 0.066
[4,   900] loss: 0.059
测试集的准确率：0.978
[5,   300] loss: 0.055
[5,   600] loss: 0.059
[5,   900] loss: 0.047
测试集的准确率：0.981
[6,   300] loss: 0.050
[6,   600] loss: 0.048
[6,   900] loss: 0.049
测试集的准确率：0.9857
[7,   300] loss: 0.041
[7,   600] loss: 0.045
[7,   900] loss: 0.046
测试集的准确率：0.9876
[8,   300] loss: 0.042
[8,   600] loss: 0.042
[8,   900] loss: 0.039
测试集的准确率：0.987
[9,   300] loss: 0.038
[9,   600] loss: 0.038
[9,   900] loss: 0.038
测试集的准确率：0.9889
[10,   300] loss: 0.035
[10,   600] loss: 0.035
[10,   900] loss: 0.037
测试集的准确率：0.9882
CPU times: total: 1min 5s
Wall time: 1min 7s
